In [1]:
import os

REPO = 'giomamaca/walmart-sales-forecasting'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    from google.colab import userdata, files

    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
        print('Upload your kaggle.json')
        uploaded = files.upload()
        shutil.move('kaggle.json', f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

    github_token = userdata.get('GITHUB_TOKEN')
    repo_dir = f'/content/{REPO.split("/")[1]}'
    if not os.path.exists(repo_dir):
        !git clone -q https://{github_token}@github.com/{REPO}.git {repo_dir}
    os.chdir(repo_dir)

    !pip install -q kaggle mlflow wandb

    import wandb
    wandb.login(key=userdata.get('WANDB_API_KEY'))

    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
    !unzip -o -q data/walmart-recruiting-store-sales-forecasting.zip -d data
    !for f in data/*.zip; do unzip -o -q "$f" -d data; done

    !mlflow db upgrade sqlite:///mlflow.db

    DATA_DIR = 'data'
else:
    DATA_DIR = '..'

Upload your kaggle.json


Saving kaggle.json to kaggle.json
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adola23 (adola23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100% 2.70M/2.70M [00:00<00:00, 181MB/s]

2026/07/09 15:10:52 INFO mlflow.store.db.utils: Updating database tables


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.base import BaseEstimator
from sklearn.pipeline import Pipeline
import mlflow
import mlflow.sklearn
import wandb

In [3]:
train = pd.read_csv(f'{DATA_DIR}/train.csv', parse_dates=['Date'])
train.shape

(421570, 5)

In [4]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.average(np.abs(y_true - y_pred), weights=weights)

In [5]:
SEQ_LEN = 52
KERNEL_SIZE = 25

class MovingAvg(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x):
        left = (self.kernel_size - 1) // 2
        right = self.kernel_size - 1 - left
        front = x[:, 0:1].repeat(1, left)
        end = x[:, -1:].repeat(1, right)
        x = torch.cat([front, x, end], dim=1)
        return self.avg(x.unsqueeze(1)).squeeze(1)

class DLinear(nn.Module):
    def __init__(self, seq_len, pred_len=1, kernel_size=25):
        super().__init__()
        self.decomp = MovingAvg(kernel_size)
        self.linear_trend = nn.Linear(seq_len, pred_len)
        self.linear_seasonal = nn.Linear(seq_len, pred_len)

    def forward(self, x):
        trend = self.decomp(x)
        seasonal = x - trend
        return self.linear_trend(trend) + self.linear_seasonal(seasonal)

In [6]:
class WindowDataset(Dataset):
    def __init__(self, windows, targets, weights):
        self.windows = windows
        self.targets = targets
        self.weights = weights

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return self.windows[idx], self.targets[idx], self.weights[idx]

def build_series_matrix(df):
    pivot = df.pivot_table(index=['Store', 'Dept'], columns='Date', values='Weekly_Sales')
    pivot = pivot.sort_index(axis=1)
    holiday = df.drop_duplicates('Date').set_index('Date')['IsHoliday'].sort_index()
    holiday = holiday.reindex(pivot.columns).fillna(False)
    return pivot, holiday

def make_windows(pivot, holiday, seq_len=SEQ_LEN):
    values = pivot.fillna(0).values
    n_series, n_time = values.shape

    windows, targets, weights = [], [], []
    for t in range(seq_len, n_time):
        window = values[:, t - seq_len:t]
        target = values[:, t]
        valid = ~pivot.iloc[:, t].isna().values
        if valid.sum() == 0:
            continue
        w = np.where(holiday.iloc[t], 5.0, 1.0)
        windows.append(window[valid])
        targets.append(target[valid])
        weights.append(np.full(valid.sum(), w))
    return np.concatenate(windows), np.concatenate(targets), np.concatenate(weights)

In [7]:
class DLinearForecaster(BaseEstimator):
    def __init__(self, seq_len=52, kernel_size=25, epochs=15, lr=1e-3, batch_size=512):
        self.seq_len = seq_len
        self.kernel_size = kernel_size
        self.epochs = epochs
        self.lr = lr
        self.batch_size = batch_size

    def fit(self, X, y, log_fn=None):
        df = X.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df['Weekly_Sales'] = y.values if hasattr(y, 'values') else y

        pivot, holiday = build_series_matrix(df)
        self.seq_len_ = min(self.seq_len, pivot.shape[1] - 1)
        self.series_mean_ = pivot.mean(axis=1).fillna(0)
        self.series_std_ = pivot.std(axis=1).fillna(1).replace(0, 1)

        norm_pivot = pivot.sub(self.series_mean_, axis=0).div(self.series_std_, axis=0)
        windows, targets, weights = make_windows(norm_pivot, holiday, self.seq_len_)

        X_t = torch.tensor(windows, dtype=torch.float32)
        y_t = torch.tensor(targets, dtype=torch.float32).unsqueeze(1)
        w_t = torch.tensor(weights, dtype=torch.float32).unsqueeze(1)

        loader = DataLoader(WindowDataset(X_t, y_t, w_t), batch_size=self.batch_size, shuffle=True)

        self.model_ = DLinear(self.seq_len_, 1, self.kernel_size)
        opt = torch.optim.Adam(self.model_.parameters(), lr=self.lr)

        for epoch in range(self.epochs):
            epoch_loss = 0.0
            for xb, yb, wb in loader:
                opt.zero_grad()
                pred = self.model_(xb)
                loss = (wb * (pred - yb).abs()).mean()
                loss.backward()
                opt.step()
                epoch_loss += loss.item() * len(xb)
            epoch_loss /= len(loader.dataset)
            if log_fn is not None:
                log_fn(epoch, epoch_loss)

        self.history_ = pivot
        return self

    def _forecast_series(self, key, target_dates):
        if key not in self.history_.index:
            return pd.Series(self.series_mean_.mean(), index=target_dates)

        mean = self.series_mean_.loc[key]
        std = self.series_std_.loc[key]
        series = self.history_.loc[key].copy()

        max_date = max(target_dates.max(), series.index.max())
        all_dates = pd.date_range(series.index.min(), max_date, freq='W-FRI')

        series = series.reindex(all_dates)
        known_mask = series.notna()
        series = series.fillna(0)

        values = ((series - mean) / std).values.tolist()

        self.model_.eval()
        with torch.no_grad():
            for i in range(len(values)):
                if i >= self.seq_len_ and not known_mask.iloc[i]:
                    window = torch.tensor([values[i - self.seq_len_:i]], dtype=torch.float32)
                    values[i] = self.model_(window).item()

        result = pd.Series(values, index=all_dates) * std + mean
        return result.reindex(target_dates)

    def predict(self, X):
        df = X.copy().reset_index(drop=True)
        df['Date'] = pd.to_datetime(df['Date'])
        preds = np.zeros(len(df))
        for (store, dept), group in df.groupby(['Store', 'Dept']):
            forecast = self._forecast_series((store, dept), pd.DatetimeIndex(group['Date'].unique()))
            preds[group.index] = forecast.reindex(group['Date']).values
        return preds

In [8]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('DLinear_Training')

2026/07/09 15:11:08 INFO mlflow.tracking.fluent: Experiment with name 'DLinear_Training' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/walmart-sales-forecasting/mlruns/4', creation_time=1783609868033, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1783609868033, lifecycle_stage='active', name='DLinear_Training', tags={}, trace_location=None, workspace='default'>

In [9]:
with mlflow.start_run(run_name='DLinear_Cleaning'):
    n_negative = int((train['Weekly_Sales'] < 0).sum())
    mlflow.log_param('negative_sales_rows', n_negative)
    mlflow.log_param('negative_sales_action', 'clip_to_zero')

    y_train = train['Weekly_Sales'].clip(lower=0)

In [10]:
with mlflow.start_run(run_name='DLinear_Windowing'):
    mlflow.log_param('seq_len', SEQ_LEN)
    mlflow.log_param('kernel_size', KERNEL_SIZE)

    pivot, holiday = build_series_matrix(train.assign(Weekly_Sales=y_train))
    windows, targets, weights = make_windows(pivot, holiday, SEQ_LEN)
    mlflow.log_metric('n_windows', len(windows))
    mlflow.log_metric('n_series', len(pivot))

len(windows), pivot.shape

(269196, (3331, 143))

In [11]:
def time_based_splits(dates, n_splits=3):
    dates = np.sort(dates.unique())
    fold_size = len(dates) // (n_splits + 1)
    splits = []
    for i in range(1, n_splits + 1):
        train_end = dates[fold_size * i - 1]
        val_end = dates[min(fold_size * (i + 1) - 1, len(dates) - 1)]
        splits.append((train_end, val_end))
    return splits

splits = time_based_splits(train['Date'])
splits

[(np.datetime64('2010-10-01T00:00:00.000000000'),
  np.datetime64('2011-06-03T00:00:00.000000000')),
 (np.datetime64('2011-06-03T00:00:00.000000000'),
  np.datetime64('2012-02-03T00:00:00.000000000')),
 (np.datetime64('2012-02-03T00:00:00.000000000'),
  np.datetime64('2012-10-05T00:00:00.000000000'))]

In [12]:
params = dict(seq_len=SEQ_LEN, kernel_size=KERNEL_SIZE, epochs=15, lr=1e-3, batch_size=512)

with mlflow.start_run(run_name='DLinear_CV'):
    mlflow.log_params(params)

    fold_scores = []
    for i, (train_end, val_end) in enumerate(splits):
        wandb.init(project='walmart-sales-forecasting', name=f'DLinear_CV_fold{i}', config=params, reinit='finish_previous')

        train_mask = train['Date'] <= train_end
        val_mask = (train['Date'] > train_end) & (train['Date'] <= val_end)

        def log_fn(epoch, loss, fold=i):
            mlflow.log_metric(f'train_loss_fold{fold}', loss, step=epoch)
            wandb.log({'epoch': epoch, 'train_loss': loss})

        model = DLinearForecaster(**params)
        model.fit(train.loc[train_mask, ['Store', 'Dept', 'Date', 'IsHoliday']], y_train[train_mask], log_fn=log_fn)

        preds = model.predict(train.loc[val_mask, ['Store', 'Dept', 'Date', 'IsHoliday']])
        score = wmae(y_train[val_mask].values, preds, train.loc[val_mask, 'IsHoliday'].values)
        fold_scores.append(score)

        mlflow.log_metric('wmae', score, step=i)
        wandb.log({'val_wmae': score})
        wandb.finish()

    mlflow.log_metric('wmae_mean', float(np.mean(fold_scores)))
    mlflow.log_metric('wmae_std', float(np.std(fold_scores)))

fold_scores

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,█▇▆▆▅▄▄▃▃▂▂▂▂▁▁
val_wmae,▁
epoch,14
train_loss,0.58191
val_wmae,3789.00724


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁
val_wmae,▁
epoch,14
train_loss,0.56854
val_wmae,3280.84604


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_wmae,▁
epoch,14
train_loss,0.68211
val_wmae,1981.56613


[np.float64(3789.007241444761),
 np.float64(3280.8460448708884),
 np.float64(1981.5661264065984)]

In [13]:
with mlflow.start_run(run_name='DLinear_Final'):
    mlflow.log_params(params)
    wandb.init(project='walmart-sales-forecasting', name='DLinear_Final', config=params, reinit='finish_previous')

    def log_fn(epoch, loss):
        mlflow.log_metric('train_loss', loss, step=epoch)
        wandb.log({'epoch': epoch, 'train_loss': loss})

    pipeline = Pipeline([
        ('model', DLinearForecaster(**params)),
    ])
    pipeline.fit(train[['Store', 'Dept', 'Date', 'IsHoliday']], y_train, model__log_fn=log_fn)

    preds = pipeline.predict(train[['Store', 'Dept', 'Date', 'IsHoliday']])
    train_wmae = wmae(y_train.values, preds, train['IsHoliday'].values)
    mlflow.log_metric('train_wmae', train_wmae)
    wandb.log({'train_wmae': train_wmae})

    mlflow.sklearn.log_model(pipeline, name='model', serialization_format='cloudpickle')
    wandb.finish()

train_wmae

2026/07/09 15:16:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/07/09 15:16:32 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_wmae,▁
epoch,14
train_loss,0.64534
train_wmae,0.0


np.float64(2.8204687984997125e-14)

In [14]:
if IN_COLAB:
    import requests
    gh_user = requests.get('https://api.github.com/user', headers={'Authorization': f'token {github_token}'}).json()['login']
    !git config user.name "{gh_user}"
    !git config user.email "{gh_user}@users.noreply.github.com"

    !git add mlflow.db mlruns
    !git commit -m "Run model_experiment_DLinear.ipynb in Colab"
    !git push

[main 6166e1e] Run model_experiment_DLinear.ipynb in Colab
 6 files changed, 61 insertions(+)
 create mode 100644 mlruns/4/models/m-28cc591d849d4b439cd95feb3d70a484/artifacts/MLmodel
 create mode 100644 mlruns/4/models/m-28cc591d849d4b439cd95feb3d70a484/artifacts/conda.yaml
 create mode 100644 mlruns/4/models/m-28cc591d849d4b439cd95feb3d70a484/artifacts/model.pkl
 create mode 100644 mlruns/4/models/m-28cc591d849d4b439cd95feb3d70a484/artifacts/python_env.yaml
 create mode 100644 mlruns/4/models/m-28cc591d849d4b439cd95feb3d70a484/artifacts/requirements.txt
Enumerating objects: 16, done.
Counting objects: 100% (16/16), done.
Delta compression using up to 2 threads
Compressing objects: 100% (11/11), done.
Writing objects: 100% (13/13), 1.78 MiB | 1.99 MiB/s, done.
Total 13 (delta 3), reused 1 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/giomamaca/walmart-sales-forecasting.git
   b41a1c8..6166e1e  main -> main
